In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

In [2]:
#!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium

In [3]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from numba import njit, prange
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import os
import json
import sys
import gc

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.preprocessing import StandardScaler

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
index_mapping = {
    # Indices
    "Nifty": {
        "symbol": "NSE:NIFTY50-INDEX",
        "quantity": 75
    },
    "Bank_Nifty": {
        "symbol": "NSE:NIFTYBANK-INDEX",
        "quantity": 30
    },
    "Finnifty": {
        "symbol": "NSE:FINNIFTY-INDEX",
        "quantity": 40
    },
    "Sensex": {
        "symbol": "BSE:SENSEX-INDEX",
        "quantity": 20
    },
    "Bankex": {
        "symbol": "BSE:BANKEX-INDEX",
        "quantity": 15
    },

    # Complete Nifty 50 Stocks

    "Adani_Enterprises_Ltd": {
        "symbol": "NSE:ADANIENT-EQ",
        "quantity": 10
    },
    "Adani_Ports__SEZ_Ltd": {
        "symbol": "NSE:ADANIPORTS-EQ",
        "quantity": 10
    },
    "Apollo_Hospitals_Enterprise_Ltd": {
        "symbol": "NSE:APOLLOHOSP-EQ",
        "quantity": 10
    },
    "Asian_Paints_Ltd": {
        "symbol": "NSE:ASIANPAINT-EQ",
        "quantity": 10
    },
    "Axis_Bank_Ltd": {
        "symbol": "NSE:AXISBANK-EQ",
        "quantity": 10
    },
    "Bajaj_Auto_Ltd": {
        "symbol": "NSE:BAJAJ-AUTO-EQ",
        "quantity": 10
    },
    "Bajaj_Finance_Ltd": {
        "symbol": "NSE:BAJFINANCE-EQ",
        "quantity": 10
    },
    "Bajaj_Finserv_Ltd": {
        "symbol": "NSE:BAJAJFINSV-EQ",
        "quantity": 10
    },
    "Bharat_Electronics_Ltd": {
        "symbol": "NSE:BEL-EQ",
        "quantity": 10
    },
    "Bharti_Airtel_Ltd": {
        "symbol": "NSE:BHARTIARTL-EQ",
        "quantity": 10
    },
    "BPCL_Ltd": {
        "symbol": "NSE:BPCL-EQ",
        "quantity": 10
    },
    "Cipla_Ltd": {
        "symbol": "NSE:CIPLA-EQ",
        "quantity": 10
    },
    "Coal_India_Ltd": {
        "symbol": "NSE:COALINDIA-EQ",
        "quantity": 10
    },
    "Dr_Reddys_Laboratories_Ltd": {
        "symbol": "NSE:DRREDDY-EQ",
        "quantity": 10
    },
    "Eicher_Motors_Ltd": {
        "symbol": "NSE:EICHERMOT-EQ",
        "quantity": 10
    },
    "Grasim_Industries_Ltd": {
        "symbol": "NSE:GRASIM-EQ",
        "quantity": 10
    },
    "HCL_Technologies_Ltd": {
        "symbol": "NSE:HCLTECH-EQ",
        "quantity": 10
    },
    "HDFC_Bank_Ltd": {
        "symbol": "NSE:HDFCBANK-EQ",
        "quantity": 10
    },
    "HDFC_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:HDFCLIFE-EQ",
        "quantity": 10
    },
    "Hero_MotoCorp_Ltd": {
        "symbol": "NSE:HEROMOTOCO-EQ",
        "quantity": 10
    },
    "Hindalco_Industries_Ltd": {
        "symbol": "NSE:HINDALCO-EQ",
        "quantity": 10
    },
    "Hindustan_Unilever_Ltd": {
        "symbol": "NSE:HINDUNILVR-EQ",
        "quantity": 10
    },
    "ICICI_Bank_Ltd": {
        "symbol": "NSE:ICICIBANK-EQ",
        "quantity": 10
    },
    "IndusInd_Bank_Ltd": {
        "symbol": "NSE:INDUSINDBK-EQ",
        "quantity": 10
    },
    "Infosys_Ltd": {
        "symbol": "NSE:INFY-EQ",
        "quantity": 10
    },
    "Indian_Oil_Corporation_Ltd": {
        "symbol": "NSE:IOC-EQ",
        "quantity": 10
    },
    "JSW_Steel_Ltd": {
        "symbol": "NSE:JSWSTEEL-EQ",
        "quantity": 10
    },
    "Kotak_Mahindra_Bank_Ltd": {
        "symbol": "NSE:KOTAKBANK-EQ",
        "quantity": 10
    },
    "Larsen__Toubro_Ltd": {
        "symbol": "NSE:LT-EQ",
        "quantity": 10
    },
    "Mahindra__Mahindra_Ltd": {
        "symbol": "NSE:M&M-EQ",
        "quantity": 10
    },
    "Maruti_Suzuki_India_Ltd": {
        "symbol": "NSE:MARUTI-EQ",
        "quantity": 10
    },
    "Nestle_India_Ltd": {
        "symbol": "NSE:NESTLEIND-EQ",
        "quantity": 10
    },
    "NTPC_Ltd": {
        "symbol": "NSE:NTPC-EQ",
        "quantity": 10
    },
    "ONGC_Ltd": {
        "symbol": "NSE:ONGC-EQ",
        "quantity": 10
    },
    "Power_Grid_Corporation_of_India_Ltd": {
        "symbol": "NSE:POWERGRID-EQ",
        "quantity": 10
    },
    "Reliance_Industries_Ltd": {
        "symbol": "NSE:RELIANCE-EQ",
        "quantity": 10
    },
    "State_Bank_of_India_(SBI)_Ltd": {
        "symbol": "NSE:SBIN-EQ",
        "quantity": 10
    },
    "SBI_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:SBILIFE-EQ",
        "quantity": 10
    },
    "Sun_Pharmaceutical_Industries_Ltd": {
        "symbol": "NSE:SUNPHARMA-EQ",
        "quantity": 10
    },
    "Tata_Consumer_Products_Ltd": {
        "symbol": "NSE:TATACONSUM-EQ",
        "quantity": 10
    },
    "Tata_Motors_Ltd": {
        "symbol": "NSE:TATAMOTORS-EQ",
        "quantity": 10
    },
    "Tata_Steel_Ltd": {
        "symbol": "NSE:TATASTEEL-EQ",
        "quantity": 10
    },
    "TCS_Ltd": {
        "symbol": "NSE:TCS-EQ",
        "quantity": 10
    },
    "Tech_Mahindra_Ltd": {
        "symbol": "NSE:TECHM-EQ",
        "quantity": 10
    },
    "Titan_Company_Ltd": {
        "symbol": "NSE:TITAN-EQ",
        "quantity": 10
    },
    "Trent_Ltd": {
        "symbol": "NSE:TRENT-EQ",
        "quantity": 10
    },
    "UltraTech_Cement_Ltd": {
        "symbol": "NSE:ULTRACEMCO-EQ",
        "quantity": 10
    },
    "Wipro_Ltd": {
        "symbol": "NSE:WIPRO-EQ",
        "quantity": 10
    }
}

In [9]:
index_mapping = {
    # Indices
    "Nifty": {
        "symbol": "NSE:NIFTY50-INDEX",
        "quantity": 75
    },
    "Bank_Nifty": {
        "symbol": "NSE:NIFTYBANK-INDEX",
        "quantity": 30
    },
    "Finnifty": {
        "symbol": "NSE:FINNIFTY-INDEX",
        "quantity": 40
    },
    "Sensex": {
        "symbol": "BSE:SENSEX-INDEX",
        "quantity": 20
    },
    "Bankex": {
        "symbol": "BSE:BANKEX-INDEX",
        "quantity": 15
    },
}

In [10]:
# Historical config for fetching and saving raw data
history_config = {
    "timeframes": [2, 3, 5, 10, 15, 20, 30, 45, 60, 120, 180, 240],
    "colab_folder_path": "/content/drive/MyDrive/historical_data",
    "colab_folder_path_processed": "processed_data",
    "local_folder_path": "historical_data",
    "local_folder_path_processed": "C:\\Users\\iamma\\OneDrive\\Desktop\\processed_data",
    "bar_limit": 10  # parameter for fetch_train_candle_data
}

In [11]:
if "google.colab" in sys.modules:
    folder_path = history_config["colab_folder_path"]
else:
    folder_path = history_config["local_folder_path"]

In [12]:
# Ensure the folder exists
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

In [13]:
# Main loop to fetch and save data
for instrument_name, info in index_mapping.items():
    for tf in history_config["timeframes"]:
        file_name = f"{instrument_name}_{tf}.csv"
        file_path = os.path.join(folder_path, file_name)

        # Skip if file already exists
        if os.path.exists(file_path):
            print(f"File already exists for {instrument_name} timeframe {tf}. Skipping.")
            continue

        # Fetch data for given bar_limit, symbol, and timeframe
        df = fetch_train_candle_data(history_config["bar_limit"], info["symbol"], tf)

        # Save if data is not empty
        if not df.empty:
            df.to_csv(file_path, index=False)
            print(f"Saved data for {instrument_name} timeframe {tf} at {file_path}")
        else:
            print(f"No data returned for {instrument_name} timeframe {tf}.")

File already exists for Nifty timeframe 2. Skipping.
File already exists for Nifty timeframe 3. Skipping.
File already exists for Nifty timeframe 5. Skipping.
File already exists for Nifty timeframe 10. Skipping.
File already exists for Nifty timeframe 15. Skipping.
File already exists for Nifty timeframe 20. Skipping.
File already exists for Nifty timeframe 30. Skipping.
File already exists for Nifty timeframe 45. Skipping.
File already exists for Nifty timeframe 60. Skipping.
File already exists for Nifty timeframe 120. Skipping.
File already exists for Nifty timeframe 180. Skipping.
File already exists for Nifty timeframe 240. Skipping.
File already exists for Bank_Nifty timeframe 2. Skipping.
File already exists for Bank_Nifty timeframe 3. Skipping.
File already exists for Bank_Nifty timeframe 5. Skipping.
File already exists for Bank_Nifty timeframe 10. Skipping.
File already exists for Bank_Nifty timeframe 15. Skipping.
File already exists for Bank_Nifty timeframe 20. Skipping.
F

In [14]:
@njit
def label_signals_jit(close_arr, high_arr, low_arr, target_arr, stoploss_arr):
    n = len(close_arr)
    signals = np.zeros(n, dtype=np.float32)
    entry_prices = np.zeros(n, dtype=np.float32)
    exit_prices = np.zeros(n, dtype=np.float32)
    candles_to_profit = np.zeros(n, dtype=np.float32)
    candles_to_loss = np.zeros(n, dtype=np.float32)

    for i in range(n - 1):
        entry_price = close_arr[i]
        target = target_arr[i]
        stop_loss = stoploss_arr[i]

        buy_target_price = entry_price + target
        buy_sl_price = entry_price - stop_loss
        sell_target_price = entry_price - target
        sell_sl_price = entry_price + stop_loss

        signal_found = False
        for offset in range(i + 1, n):
            future_high = high_arr[offset]
            future_low = low_arr[offset]

            triggers = []
            # Check Buy triggers
            if future_high >= buy_target_price:
                triggers.append((1, offset - i))  # buy target
            if future_low <= buy_sl_price:
                triggers.append((2, offset - i))  # buy stoploss
            # Check Sell triggers
            if future_low <= sell_target_price:
                triggers.append((3, offset - i))  # sell target
            if future_high >= sell_sl_price:
                triggers.append((4, offset - i))  # sell stoploss

            # If any triggers fired, pick the first based on priority (1,2,3,4)
            if triggers:
                triggers.sort(key=lambda x: x[0])
                first_trigger, candle_count = triggers[0]

                signals[i] = float(first_trigger)
                entry_prices[i] = entry_price

                if first_trigger == 1:  # buy target
                    exit_prices[i] = buy_target_price
                    candles_to_profit[i] = float(candle_count)
                elif first_trigger == 2:  # buy stoploss
                    exit_prices[i] = buy_sl_price
                    candles_to_loss[i] = float(candle_count)
                elif first_trigger == 3:  # sell target
                    exit_prices[i] = sell_target_price
                    candles_to_profit[i] = float(candle_count)
                elif first_trigger == 4:  # sell stoploss
                    exit_prices[i] = sell_sl_price
                    candles_to_loss[i] = float(candle_count)

                signal_found = True
                break

        if not signal_found:
            signals[i] = 0
            entry_prices[i] = entry_price

    return signals, entry_prices, exit_prices, candles_to_profit, candles_to_loss


class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns: [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')
        self.df['datetime'] = (
            pd.to_datetime(self.df['datetime'], unit='s')
            .dt.tz_localize('UTC')
            .dt.tz_convert(ist)
            .dt.tz_localize(None)
        )

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        #if 'volume' in self.df.columns:
        #   vol_condition = self.df['volume'].isnull() | (self.df['volume'] <= 0)
        #    if vol_condition.any():
        #        self.df.drop('volume', axis=1, inplace=True)

        self.df.drop('volume', axis=1, inplace=True)

        return self

    def add_indicator_features(self):
        # ATR
        self.df['atr_14'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)

        # RSI
        self.df['rsi_14'] = ta.rsi(
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)

        # Stoch
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14,
            d=3
        ).astype(np.float32).round(2)
        self.df = self.df.join(stoch)

        # MACD
        macd = ta.macd(
            close=self.df['close'],
            fast=12,
            slow=26,
            signal=9
        ).astype(np.float32).round(2)
        self.df = self.df.join(macd)

        # ADX
        adx = ta.adx(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)
        self.df = self.df.join(adx[['ADX_14']])
        self.df.rename(columns={'ADX_14': 'adx_14'}, inplace=True)

        # EMAs
        self.df['ema_50'] = ta.ema(self.df['close'], length=50).astype(np.float32).round(2)
        self.df['ema_200'] = ta.ema(self.df['close'], length=200).astype(np.float32).round(2)

        # SMA
        self.df['sma_20'] = ta.sma(self.df['close'], length=20).astype(np.float32).round(2)

        # Keltner Channels
        kc = ta.kc(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=20
        ).astype(np.float32).round(2)
        self.df = self.df.join(kc)


        # Price Action Features
        self.df['price_range'] = (self.df['high'] - self.df['low']).astype(np.float32).round(2)
        self.df['body_size'] = (self.df['close'] - self.df['open']).abs().astype(np.float32).round(2)
        self.df['upper_wick'] = (
            self.df['high'] - self.df[['open', 'close']].max(axis=1)
        ).astype(np.float32).round(2)
        self.df['lower_wick'] = (
            self.df[['open', 'close']].min(axis=1) - self.df['low']
        ).astype(np.float32).round(2)

        return self

    def add_time_features(self):
        # Hour, day_of_week, month
        self.df['hour'] = self.df.index.hour.astype(np.int32)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(np.int32)
        self.df['month'] = self.df.index.month.astype(np.int32)
        return self

    def add_adaptive_targets_and_stops(self):
        # ATR-based target and stoploss
        self.df['Target'] = (4.0 * self.df['atr_14']).astype(np.float32).round(2)
        self.df['StopLoss'] = (2.0 * self.df['atr_14']).astype(np.float32).round(2)
        return self

    def label_signals(self):
        from __main__ import label_signals_jit  # or from the same notebook cell, as needed

        close_arr = self.df['close'].values
        high_arr = self.df['high'].values
        low_arr = self.df['low'].values
        target_arr = self.df['Target'].values
        stoploss_arr = self.df['StopLoss'].values

        signals, entry_prices, exit_prices, ctp, ctl = label_signals_jit(
            close_arr,
            high_arr,
            low_arr,
            target_arr,
            stoploss_arr
        )
        self.df['Signal'] = signals.astype(np.float32)
        self.df['Entry Price'] = entry_prices.astype(np.float32).round(2)
        self.df['Exit Price'] = exit_prices.astype(np.float32).round(2)
        self.df['candles_to_profit'] = ctp.astype(np.float32)
        self.df['candles_to_loss'] = ctl.astype(np.float32)

        return self

    def run_pipeline(self):
        (self.preprocess_datetime()
             .clean_data()
             .add_indicator_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
             .label_signals())
        return self

    def get_processed_df(self):
        # Drop rows with any NaN
        self.df.dropna(axis=0, how='any', inplace=True)

        # Remove Entry/Exit Price columns if you don't need them
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        return self.df

In [15]:
if "google.colab" in sys.modules:
    processed_folder_path = history_config["colab_folder_path_processed"]
else:
    processed_folder_path = history_config["local_folder_path_processed"]

if not os.path.exists(processed_folder_path):
    os.makedirs(processed_folder_path)

In [16]:
# We'll store (instrument_name, timeframe, processed_df) tuples in this list
temp_data_list = []

# Convert index_mapping keys into a set for quick lookup
valid_instruments = set(index_mapping.keys())

# Loop through files in the historical folder
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)

        # Extract instrument name and timeframe
        instrument_name, timeframe_str = file.replace(".csv", "").rsplit("_", 1)
        
        # Ensure the instrument is in index_mapping
        if instrument_name not in valid_instruments:
            print(f"Skipping {file} as {instrument_name} is not in index_mapping.")
            continue

        # Convert timeframe string to integer
        try:
            timeframe = int(timeframe_str)
        except ValueError:
            print(f"Skipping {file} due to invalid timeframe format.")
            continue

        # Ensure timeframe is in history_config
        if timeframe not in history_config["timeframes"]:
            print(f"Skipping {file} as timeframe {timeframe} is not in config.")
            continue

        processed_file_name = f"{instrument_name}_{timeframe}_processed.csv"
        processed_file_path = os.path.join(processed_folder_path, processed_file_name)

        # Skip if already processed
        if os.path.exists(processed_file_path):
            print(f"Processed file already exists for {instrument_name} {timeframe}M. Skipping.")
        else:
            # Process file
            raw_df = pd.read_csv(file_path).drop_duplicates(subset='datetime', keep='first')
            pipeline = FullFeaturePipeline(raw_df)
            pipeline.run_pipeline()
            processed_df = pipeline.get_processed_df()

            processed_df.to_csv(processed_file_path, index=True, index_label='datetime')
            print(f"Processed and saved data for {instrument_name} {timeframe}M at {processed_file_path}")

            # Cleanup to free memory
            del raw_df
            del processed_df
            gc.collect()

        # Store processed data
        temp_data_list.append((instrument_name, timeframe, processed_file_path))

Processed file already exists for Bankex 10M. Skipping.
Processed file already exists for Bankex 120M. Skipping.
Processed file already exists for Bankex 15M. Skipping.
Processed file already exists for Bankex 180M. Skipping.
Processed file already exists for Bankex 2M. Skipping.
Processed file already exists for Bankex 20M. Skipping.
Processed file already exists for Bankex 240M. Skipping.
Processed file already exists for Bankex 3M. Skipping.
Processed file already exists for Bankex 30M. Skipping.
Processed file already exists for Bankex 45M. Skipping.
Processed file already exists for Bankex 5M. Skipping.
Processed file already exists for Bankex 60M. Skipping.
Processed file already exists for Bank_Nifty 10M. Skipping.
Processed file already exists for Bank_Nifty 120M. Skipping.
Processed file already exists for Bank_Nifty 15M. Skipping.
Processed file already exists for Bank_Nifty 180M. Skipping.
Processed file already exists for Bank_Nifty 2M. Skipping.
Processed file already exis

In [17]:
# Sort by timeframe descending, then by instrument name alphabetically
temp_data_list.sort(key=lambda x: (-x[1], x[0]))

instruments_config = []
for i, (instrument_name, timeframe, csv_path) in enumerate(temp_data_list):
    lot_size = index_mapping[instrument_name]["quantity"]
    dynamic_name = f"{instrument_name.upper()}_{timeframe}M"

    instruments_config.append({
        "name": dynamic_name,
        "file_path": csv_path,
        "lot_size": lot_size,
        "transaction_cost": 20.0  # default brokerage
    })

In [18]:
# Now build the main config dictionary
config = {
    "instruments": instruments_config,
    "window_size": 10,
    "initial_capital": 500000,
    "unrealized_pnl_weight": 0.1,
    "time_penalty_weight": 0.001,
    "volatility_penalty_weight": 0.05,
    "target_bonus": 0.5,
    "sl_penalty": 0.5,
    "misalignment_penalty": 0.5,
    "missed_opportunity_penalty": 0.05,
    "target_proximity_weight": 0.2,
    "sl_proximity_weight": 0.05,
    "max_step_reward": 5,

    # Trailing Stop Parameters
    "enable_trailing_sl": True,
    "profit_lock_threshold": 0.5,
    "smart_exit_bonus": 0.3,
    "trailing_exit_bonus": 0.4,

    "initial_cap_multiplier": 10,   # factor for initial capital
    "normalize_data": True,         # whether to normalize data for the agent's observation
}

# Margin constants
MARGIN_FRACTION = 0.05
MARGIN_CALL_THRESHOLD = 0.5

In [19]:
# Print a summary of the dynamically created instruments
print("Final Config Instruments:")
for inst in config["instruments"]:
    print(f"{inst['name']}: lot_size={inst['lot_size']}, brokerage={inst['transaction_cost']}")

Final Config Instruments:
BANK_NIFTY_240M: lot_size=30, brokerage=20.0
BANKEX_240M: lot_size=15, brokerage=20.0
FINNIFTY_240M: lot_size=40, brokerage=20.0
NIFTY_240M: lot_size=75, brokerage=20.0
SENSEX_240M: lot_size=20, brokerage=20.0
BANK_NIFTY_180M: lot_size=30, brokerage=20.0
BANKEX_180M: lot_size=15, brokerage=20.0
FINNIFTY_180M: lot_size=40, brokerage=20.0
NIFTY_180M: lot_size=75, brokerage=20.0
SENSEX_180M: lot_size=20, brokerage=20.0
BANK_NIFTY_120M: lot_size=30, brokerage=20.0
BANKEX_120M: lot_size=15, brokerage=20.0
FINNIFTY_120M: lot_size=40, brokerage=20.0
NIFTY_120M: lot_size=75, brokerage=20.0
SENSEX_120M: lot_size=20, brokerage=20.0
BANK_NIFTY_60M: lot_size=30, brokerage=20.0
BANKEX_60M: lot_size=15, brokerage=20.0
FINNIFTY_60M: lot_size=40, brokerage=20.0
NIFTY_60M: lot_size=75, brokerage=20.0
SENSEX_60M: lot_size=20, brokerage=20.0
BANK_NIFTY_45M: lot_size=30, brokerage=20.0
BANKEX_45M: lot_size=15, brokerage=20.0
FINNIFTY_45M: lot_size=40, brokerage=20.0
NIFTY_45M: lo

In [20]:
def get_dataframe_by_name(instrument_name, config):
    # instrument_name is something like "BANKNIFTY_5M", "NIFTY50_15M", etc.
    for instrument in config["instruments"]:
        if instrument["name"] == instrument_name:
            return pd.read_csv(instrument["file_path"], parse_dates=['datetime'], index_col='datetime')
    return None


my_df = get_dataframe_by_name("BANK_NIFTY_5M", config)

my_df

,open,high,low,close,atr_14,rsi_14,STOCHk_14_3_3,STOCHd_14_3_3,MACD_12_26_9,MACDh_12_26_9,...,upper_wick,lower_wick,hour,day_of_week,month,Target,StopLoss,Signal,candles_to_profit,candles_to_loss
datetime,,,,,,,,,,,,,,,,,,,,,
2022-03-07 14:05:00,32532.20,32568.60,32471.80,32484.80,101.15,27.20,8.27,7.91,-147.59,-19.05,...,36.40,13.00,14,0,3,404.60,202.30,4.0,0.0,6.0
2022-03-07 14:10:00,32479.30,32510.50,32428.20,32466.40,99.80,26.56,4.92,6.94,-153.20,-19.73,...,31.20,38.20,14,0,3,399.20,199.60,4.0,0.0,4.0
2022-03-07 14:15:00,32467.60,32528.70,32376.50,32452.10,103.54,26.05,7.60,6.93,-156.99,-18.81,...,61.10,75.60,14,0,3,414.16,207.08,4.0,0.0,3.0
2022-03-07 14:20:00,32451.90,32599.40,32424.70,32544.40,108.63,34.82,17.75,10.09,-150.80,-10.10,...,55.00,27.20,14,0,3,434.52,217.26,4.0,0.0,6.0
2022-03-07 14:25:00,32542.80,32632.20,32528.20,32612.50,108.30,40.44,30.94,18.76,-138.81,1.51,...,19.70,14.60,14,0,3,433.20,216.60,4.0,0.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-27 15:05:00,52306.20,52351.10,52285.05,52295.50,44.00,39.75,27.14,25.89,-5.00,-11.11,...,44.90,10.45,15,2,11,176.00,88.00,0.0,0.0,0.0
2024-11-27 15:10:00,52294.50,52348.95,52292.60,52302.00,44.88,41.58,28.88,28.26,-7.16,-10.62,...,46.95,1.90,15,2,11,179.52,89.76,0.0,0.0,0.0
2024-11-27 15:15:00,52300.30,52305.55,52271.30,52285.25,44.12,38.35,23.22,26.41,-10.12,-10.86,...,5.25,13.95,15,2,11,176.48,88.24,0.0,0.0,0.0


In [21]:
class Position:
    def __init__(self):
        self.reset()

    def reset(self):
        self.status = "flat"
        self.direction = 0
        self.entry_price = 0.0
        self.sl_points = 0.0
        self.target_points = 0.0
        self.quantity = 0
        self.time_in_trade = 0
        self.margin_used = 0.0
        self.unrealized_pnl = 0.0
        self.trailing_sl_price = 0.0

    def open(self, entry_price, sl_points, target_points, direction, quantity):
        self.status = "long" if direction > 0 else "short"
        self.direction = direction
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.quantity = quantity
        self.time_in_trade = 0
        self.trailing_sl_price = entry_price - (sl_points * direction)

    def update_unrealized_pnl(self, current_price):
        self.unrealized_pnl = (current_price - self.entry_price) * self.direction * self.quantity
        return self.unrealized_pnl

    def update_trailing_sl(self, current_price):
        """Update trailing stop loss if price moves favorably"""
        if self.status == "flat":
            return False
        
        # For long positions
        if self.direction == 1:
            # Calculate new potential stop loss level (moves up with price)
            potential_sl = current_price - self.sl_points
            # Only update if the new SL would be higher than current SL
            if potential_sl > self.trailing_sl_price:
                self.trailing_sl_price = potential_sl
                return True
        # For short positions
        elif self.direction == -1:
            # Calculate new potential stop loss level (moves down with price)
            potential_sl = current_price + self.sl_points
            # Only update if the new SL would be lower than current SL
            if potential_sl < self.trailing_sl_price:
                self.trailing_sl_price = potential_sl
                return True
        
        return False

In [22]:
class TradingEnv(gym.Env):
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        self.config = env_config
        self.total_reward = 0.0

        # Mandatory configuration parameters
        self.window_size = self.config["window_size"]
        self.lot_size = self.config["lot_size"]
        self.transaction_cost = self.config["transaction_cost"]
        self.instrument_name = self.config.get("name", "Unknown")
        self.instrument_id = self.config.get("instrument_id", 0)
        self.normalize_data = self.config.get("normalize_data", False)
        self.csv_path = self.config["file_path"]

        # For chunked loading to save memory
        self.chunk_size = self.config.get("chunk_size", 10000)
        self.data_iterator = pd.read_csv(
            self.csv_path,
            parse_dates=['datetime'],
            index_col='datetime',
            chunksize=self.chunk_size
        )
        self.done_loading = False
        self._load_next_chunk(initial=True)

        # Initial capital and trading variables
        self.initial_capital = self.config.get("initial_capital")
        self.current_capital = None
        self.current_step = None

        self.position = Position()
        self.trade_log = []

        # Enhanced action space for close position action
        # 0: Hold, 1: Long, 2: Short, 3: Close Position
        self.action_space = spaces.Discrete(4)
        
        # Define observation space
        n_features = self.buffer_obs.shape[1]
        self.observation_space = spaces.Dict({
            "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
            "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32),
            "instrument_info": spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
        })

        # Reward shaping parameters (unchanged)
        self.unrealized_pnl_weight = self.config.get("unrealized_pnl_weight", 0.1)
        self.time_penalty_weight = self.config.get("time_penalty_weight", 0.02)
        self.volatility_penalty_weight = self.config.get("volatility_penalty_weight", 0.1)
        self.target_bonus = self.config.get("target_bonus", 0.5)
        self.sl_penalty = self.config.get("sl_penalty", 0.5)
        self.misalignment_penalty = self.config.get("misalignment_penalty", 0.5)
        self.missed_opportunity_penalty = self.config.get("missed_opportunity_penalty", 0.05)
        self.target_proximity_weight = self.config.get("target_proximity_weight", 0.2)
        self.sl_proximity_weight = self.config.get("sl_proximity_weight", 0.05)
        self.max_step_reward = self.config.get("max_step_reward", 5.0)
        
        self.enable_trailing_sl = self.config.get("enable_trailing_sl", True)
        self.profit_lock_threshold = self.config.get("profit_lock_threshold", 0.5)  # % of target to trigger trailing
        self.smart_exit_bonus = self.config.get("smart_exit_bonus", 0.3)  # Bonus for exiting losing trades early
        self.trailing_exit_bonus = self.config.get("trailing_exit_bonus", 0.4)  # Bonus for trailing profitable trades

    def _load_next_chunk(self, initial=False):
        """
        Loads the next chunk of data from the CSV. To preserve continuity,
        if this isn’t the initial load, the last `window_size` rows from the previous
        chunk are kept and prepended to the new chunk.
        This method is used only when not training on TPU.
        """
        try:
            next_chunk = next(self.data_iterator)
        except StopIteration:
            self.done_loading = True
            return

        if not initial:
            # Keep an overlap equal to window_size for continuity
            overlap = self.window_size
            tail = self.data.iloc[-overlap:]
            self.data = pd.concat([tail, next_chunk])
        else:
            self.data = next_chunk

        # Update total rows and buffers based on the new chunk
        self.total_rows = len(self.data)
        self.buffer_raw = self.data.drop(
            columns=['Signal', 'candles_to_profit', 'candles_to_loss'], errors='ignore'
        ).copy()
        if 'Signal' in self.data.columns:
            self.buffer_guidance = self.data[['Signal', 'candles_to_profit', 'candles_to_loss']].copy()
        else:
            self.buffer_guidance = pd.DataFrame(index=self.data.index)
        self.buffer_obs = self.buffer_raw.copy()

        # Preprocess buffers
        self.buffer_raw = self.buffer_raw.ffill().bfill()
        self.buffer_guidance = self.buffer_guidance.ffill().bfill()
        self.buffer_obs = self.buffer_obs.ffill().bfill()

        if self.normalize_data:
            self.scaler = StandardScaler()
            self.scaler.fit(self.buffer_obs.values)
            self.buffer_obs = pd.DataFrame(
                self.scaler.transform(self.buffer_obs.values),
                columns=self.buffer_obs.columns,
                index=self.buffer_obs.index
            )
        else:
            self.scaler = None

    def reset(self, seed=None, options=None):
        """Resets the environment to its initial state."""
        super().reset(seed=seed)
        self.current_step = self.window_size
        if self.current_step >= self.total_rows:
            # If current chunk is too small, try loading the next chunk
            self._load_next_chunk()
            self.current_step = self.window_size if self.total_rows > self.window_size else 0

        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log.clear()
        self.total_reward = 0.0

        obs = self._get_observation()
        return obs, {}

    def step(self, action: int):
        """
        Executes one time step in the environment.
        If we are nearing the end of the current chunk, attempt to load the next one.
        """
        if self.current_step >= self.total_rows - self.window_size and not self.done_loading:
            self._load_next_chunk()

        current_data = self._get_row(self.buffer_raw, self.current_step)
        guidance_row = self._get_row(self.buffer_guidance, self.current_step)
        reward = 0.0

        # Apply guidance reward when flat
        if self.position.status == "flat":
            reward += self._apply_guidance_reward(action, guidance_row)

        # Check for margin call
        if self.position.status != "flat":
            if self._check_margin_call(current_data['close']):
                reward += self._close_position(current_data, "Margin Call")

        # First check for stop loss hit (failsafe mechanism)
        hit_sl, fill_price = self._check_sl_hit(current_data)
        if hit_sl:
            reward += self._close_position(current_data, "SL Hit", override_exit_price=fill_price)
        else:
            # Check for target hit
            hit_target, target_fill_price = self._check_target_hit(current_data)
            if hit_target:
                reward += self._close_position(current_data, "Target Hit", override_exit_price=target_fill_price)
            # Update trailing stop loss if enabled and in a position
            elif self.enable_trailing_sl and self.position.status != "flat":
                # Check if we've reached profit lock threshold
                unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
                risk = self.position.sl_points * self.lot_size
                if risk > 0 and unrealized_pnl / risk > self.profit_lock_threshold:
                    # Update trailing stop loss
                    sl_updated = self.position.update_trailing_sl(current_data['close'])
                    if sl_updated:
                        # Small reward for successful trailing adjustment
                        reward += 0.1 * self.trailing_exit_bonus

        # Execute the action
        reward += self._execute_action(action, current_data)

        # Calculate ongoing position reward if still in a position
        if self.position.status != "flat":
            self.position.time_in_trade += 1
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
            reward += self._calculate_reward(unrealized_pnl, current_data)

        # Check for end of episode
        self.current_step += 1
        if self.current_step >= self.total_rows or self.current_capital <= (self.initial_capital / 2):
            done = True
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")
        else:
            done = False

        self.total_reward += reward
        obs = self._get_observation()
        return obs, reward, done, False, {}

    def _execute_action(self, action: int, current_data: pd.Series):
        """Executes the specified action if conditions are met."""
        reward = 0.0
        price = current_data.get('open', 0.0)
        margin_cost = price * self.lot_size * MARGIN_FRACTION

        # Action 3: Close Position (new autonomous closing capability)
        if action == 3 and self.position.status != "flat":
            unrealized_pnl = self.position.update_unrealized_pnl(price)
            risk = self.position.sl_points * self.lot_size
            
            # Smart exit logic: reward is proportional to avoiding loss
            if unrealized_pnl < 0:
                # Reward for cutting losses early
                potential_loss = min(0, unrealized_pnl)
                max_loss = -risk
                # Reward is higher when exiting earlier (smaller loss vs max possible loss)
                loss_prevented = (max_loss - abs(potential_loss)) / max_loss if max_loss > 0 else 0
                exit_reward = self.smart_exit_bonus * loss_prevented
                reason = "Smart Loss Exit"
            else:
                # Reward for securing profits
                target_pnl = self.position.target_points * self.lot_size
                profit_ratio = unrealized_pnl / target_pnl if target_pnl > 0 else 0
                exit_reward = self.trailing_exit_bonus * profit_ratio
                reason = "Smart Profit Exit"
                
            reward += self._close_position(current_data, reason) + exit_reward
            
        # Action 1: Long
        elif action == 1 and self.position.status == "flat":
            if self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("long", current_data)
        
        # Action 2: Short
        elif action == 2 and self.position.status == "flat":
            if self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("short", current_data)
        
        # Action 0: Hold
        # Does nothing
        
        return reward

    def _open_position(self, direction: str, current_data: pd.Series):
        """Opens a new trading position."""
        entry_price = current_data.get('open', 0.0)
        margin_cost = entry_price * self.lot_size * MARGIN_FRACTION
        self.current_capital -= (margin_cost + self.transaction_cost)
        self.position.margin_used = margin_cost

        sl_val = current_data.get('StopLoss', 0.0)
        tg_val = current_data.get('Target', 0.0)
        direction_int = 1 if direction == "long" else -1

        self.position.open(
            entry_price=entry_price,
            sl_points=sl_val,
            target_points=tg_val,
            direction=direction_int,
            quantity=self.lot_size
        )

        self.trade_log.append({
            "entry_time": current_data.name,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None,
            "initial_capital": self._format_currency(self.initial_capital),
            "points": None,
            "profit": None,
            "win": None,
            "reward": None,
            "time_in_trade": None,
            "trailing_exit": None
        })

    def _close_position(self, current_data: pd.Series, reason: str, override_exit_price=None) -> float:
        """Closes an active position and updates capital and trade log."""
        if self.position.status == "flat":
            return 0.0

        direction = self.position.direction
        exit_price = override_exit_price if override_exit_price is not None else current_data.get('open', 0.0)
        realized_pnl = (exit_price - self.position.entry_price) * direction * self.lot_size
        margin_return = self.position.margin_used
        self.current_capital += (margin_return + realized_pnl - self.transaction_cost)

        trade_return = realized_pnl / self.initial_capital
        total_reward = trade_return
        points = (exit_price - self.position.entry_price) * direction
        formatted_profit = self._format_currency(realized_pnl)
        win = realized_pnl > 0
        
        # Check if this was a trailing exit
        used_trailing = False
        if reason == "SL Hit" and self.position.trailing_sl_price != (self.position.entry_price - self.position.sl_points * direction):
            reason = "Trailing SL Hit"
            used_trailing = True

        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason,
                    "time_in_trade": self.position.time_in_trade,
                    "points": round(points, 2),
                    "profit": formatted_profit,
                    "win": win,
                    "reward": total_reward,
                    "trailing_exit": used_trailing
                })
                break

        self.position.reset()
        return total_reward

    def _check_sl_hit(self, current_data: pd.Series):
        """Checks if the stop loss has been hit."""
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        current_open = current_data.get('open', 0.0)
        current_high = current_data.get('high', 0.0)
        current_low = current_data.get('low', 0.0)

        # Use trailing stop loss price instead of fixed stop loss
        sl_price = self.position.trailing_sl_price
        
        if direction == 1:  # Long
            if current_open <= sl_price:
                return True, current_open
            elif current_low <= sl_price:
                return True, sl_price
        else:  # Short
            if current_open >= sl_price:
                return True, current_open
            elif current_high >= sl_price:
                return True, sl_price
        return False, None

    def _check_target_hit(self, current_data: pd.Series):
        """Checks if the target price has been hit."""
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        entry_price = self.position.entry_price
        target_points = self.position.target_points
        current_open = current_data.get('open', 0.0)
        current_high = current_data.get('high', 0.0)
        current_low = current_data.get('low', 0.0)

        if direction == 1:  # Long
            target_price = entry_price + target_points
            if current_open >= target_price:
                return True, current_open
            elif current_high >= target_price:
                return True, target_price
        else:  # Short
            target_price = entry_price - target_points
            if current_open <= target_price:
                return True, current_open
            elif current_low <= target_price:
                return True, target_price
        return False, None

    def _check_margin_call(self, current_price: float) -> bool:
        """Checks if a margin call is triggered."""
        if self.position.status == "flat":
            return False
        maintenance_margin = self.position.margin_used * MARGIN_CALL_THRESHOLD
        unrealized_pnl = self.position.update_unrealized_pnl(current_price)
        equity = self.current_capital + unrealized_pnl
        return equity < maintenance_margin

    def _apply_guidance_reward(self, action, guidance_row):
        """Applies reward based on guidance signals when flat."""
        if guidance_row.empty:
            return 0.0
        signal = guidance_row.get('Signal', 0)
        candles_to_profit = max(guidance_row.get('candles_to_profit', 1.0), 1.0)

        if signal == 1 and action == 1:  # Long signal
            return self.target_bonus / candles_to_profit
        elif signal == 3 and action == 2:  # Short signal
            return self.target_bonus / candles_to_profit
        elif signal in [1, 3] and action != (1 if signal == 1 else 2):
            return -self.misalignment_penalty
        return 0.0

    def _calculate_reward(self, unrealized_pnl: float, current_data: pd.Series) -> float:
        """Calculates step reward for an open position."""
        risk = self.position.sl_points * self.lot_size
        risk_adj_pnl = (unrealized_pnl / risk) * self.unrealized_pnl_weight if risk > 0 else 0.0
        time_penalty = self.time_penalty_weight * (self.position.time_in_trade / 100.0)

        hi, lo, op = current_data.get('high', 0.0), current_data.get('low', 0.0), current_data.get('open', 1.0)
        volatility_penalty = self.volatility_penalty_weight * ((hi - lo) / max(abs(op), 1e-6))

        if self.position.direction == 1:
            sl_price = self.position.trailing_sl_price
            sl_distance = max(0, current_data.get('close', 0.0) - sl_price)
            sl_range = self.position.sl_points
        else:
            sl_price = self.position.trailing_sl_price
            sl_distance = max(0, sl_price - current_data.get('close', 0.0))
            sl_range = self.position.sl_points
        sl_prox_ratio = min(sl_distance / sl_range, 1.0) if sl_range > 1e-6 else 1.0
        sl_proximity_penalty = self.sl_proximity_weight * (1.0 - sl_prox_ratio)

        target_proximity_bonus = 0.0
        if self.position.target_points > 0:
            target_price = self.position.entry_price + (self.position.direction * self.position.target_points)
            target_range = abs(self.position.target_points)
            close = current_data.get('close', 0.0)
            distance_to_target = max(0, (target_price - close) * self.position.direction)
            if (self.position.direction == 1 and current_data.get('high', 0.0) >= target_price) or \
               (self.position.direction == -1 and current_data.get('low', 0.0) <= target_price):
                target_proximity_bonus -= self.missed_opportunity_penalty
            if target_range > 1e-6:
                target_proximity_bonus += (1.0 - (distance_to_target / target_range)) * self.target_proximity_weight

        total_reward = risk_adj_pnl + target_proximity_bonus - time_penalty - volatility_penalty - sl_proximity_penalty
        return float(np.clip(total_reward, -self.max_step_reward, self.max_step_reward))

    def _get_observation(self) -> dict:
        """Returns the current observation."""
        idx = self.current_step
        row_obs = self._get_row(self.buffer_obs, idx).values.astype(np.float32)
        current_close = self._get_row(self.buffer_raw, idx)['close']
        unrealized_pnl = self.position.update_unrealized_pnl(current_close)
        
        # Enhanced position context with trailing stop loss information
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            unrealized_pnl / self.initial_capital,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        history_data = np.array([
            self._get_row(self.buffer_obs, i).values.astype(np.float32) if 0 <= i < self.total_rows
            else np.zeros_like(row_obs)
            for i in range(idx - self.window_size, idx)
        ], dtype=np.float32)

        instrument_info = np.array([self.lot_size / 1000.0, self.transaction_cost / 100.0], dtype=np.float32)
        return {
            "market": row_obs,
            "position": position_context,
            "history": history_data,
            "instrument_info": instrument_info
        }

    def _get_row(self, df: pd.DataFrame, index: int) -> pd.Series:
        """Retrieves a row from a DataFrame by index, handling out-of-bounds cases."""
        if index < 0 or index >= len(df) or df.empty:
            return pd.Series(np.zeros(df.shape[1] if df.shape[1] else 0, dtype=np.float32), index=df.columns if not df.empty else [])
        return df.iloc[index]

    def get_metrics(self) -> dict:
        """Returns performance metrics based on trade log."""
        if not self.trade_log:
            return {"instrument": self.instrument_name, "message": "No trades taken"}

        closed_trades = [t for t in self.trade_log if t['exit_time'] is not None]
        total_trades = len(closed_trades)
        if total_trades == 0:
            return {"instrument": self.instrument_name, "message": "No trades closed yet"}

        winning_trades = [t for t in closed_trades if t['pnl'] > 0]
        losing_trades = [t for t in closed_trades if t['pnl'] <= 0]
        win_rate = (len(winning_trades) / total_trades) * 100.0
        sum_winning = sum(t['pnl'] for t in winning_trades)
        sum_losing = sum(t['pnl'] for t in losing_trades)
        profit_factor = sum_winning / abs(sum_losing) if sum_losing != 0 else float('inf')

        # New metrics for autonomous closings
        smart_exits = [t for t in closed_trades if t['reason'] in ["Smart Loss Exit", "Smart Profit Exit"]]
        smart_exit_rate = (len(smart_exits) / total_trades) * 100.0 if total_trades > 0 else 0
        trailing_exits = [t for t in closed_trades if t['trailing_exit'] is True]
        trailing_exit_rate = (len(trailing_exits) / total_trades) * 100.0 if total_trades > 0 else 0

        return {
            "instrument": self.instrument_name,
            "initial_capital": self._format_currency(self.initial_capital),
            "final_capital": self._format_currency(self.current_capital),
            "total_trades": total_trades,
            "win_rate": f"{round(win_rate, 2)}%",
            "total_reward": round(self.total_reward, 2),
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": round(profit_factor, 2),
            "smart_exit_rate": f"{round(smart_exit_rate, 2)}%",
            "trailing_exit_rate": f"{round(trailing_exit_rate, 2)}%"
        }

    def _compute_max_drawdown(self) -> float:
        """Calculates the maximum drawdown from the equity curve."""
        equity_curve = [self.initial_capital]
        for t in self.trade_log:
            if t['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + t['pnl'])
        equity_curve = np.array(equity_curve, dtype=np.float32)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return float(dd.max())

    def _format_currency(self, value: float) -> str:
        """Formats a value into a currency string with appropriate units."""
        abs_value = abs(value)
        if abs_value >= 1e7:
            return f"₹{value / 1e7:.2f}Cr"
        elif abs_value >= 1e5:
            return f"₹{value / 1e5:.2f}L"
        elif abs_value >= 1e3:
            return f"₹{value / 1e3:.2f}K"
        else:
            return f"₹{value:.2f}"

    def render(self, mode='human'):
        """Renders the current state of the environment."""
        if self.position.status != "flat":
            position_info = f"Position: {self.position.status}, Entry: {self.position.entry_price:.2f}, " \
                            f"Trailing SL: {self.position.trailing_sl_price:.2f}, " \
                            f"Target: {self.position.entry_price + (self.position.direction * self.position.target_points):.2f}"
            return f"Step: {self.current_step}, Capital: {self._format_currency(self.current_capital)}, {position_info}"
        else:
            return f"Step: {self.current_step}, Capital: {self._format_currency(self.current_capital)}, Position: Flat"

In [23]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from tqdm import tqdm

# Model configuration
model_config = {
    "history_len": config["window_size"],  # Assume config is defined elsewhere
    "n_features": None,  # Will be set after inspecting an env's observation
    "d_model": 64,      # Hidden dimension
    "n_heads": 2,        # Number of heads
    "n_layers": 1,       # Number of transformer layers
}

In [24]:
class TransformerPolicy(nn.Module):
    def __init__(self, history_len, n_features, d_model=256, n_heads=8, n_layers=4):
        super(TransformerPolicy, self).__init__()
        self.history_len = history_len
        self.n_features = n_features

        # Embedding layers
        self.market_embed = nn.Linear(n_features, d_model)
        self.position_embed = nn.Linear(4, d_model)
        self.instrument_embed = nn.Linear(2, d_model)
        self.history_embed = nn.Linear(n_features, d_model)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Output head
        self.fc_out = nn.Linear(d_model, 3)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, obs):
        device = next(self.parameters()).device

        # Create tensors on the model's device
        market = torch.tensor(obs["market"], dtype=torch.float32, device=device).unsqueeze(0)
        position = torch.tensor(obs["position"], dtype=torch.float32, device=device).unsqueeze(0)
        history = torch.tensor(obs["history"], dtype=torch.float32, device=device).unsqueeze(0)
        instrument = torch.tensor(obs["instrument_info"], dtype=torch.float32, device=device).unsqueeze(0)

        # Embed components
        market_emb = self.market_embed(market).unsqueeze(1)
        position_emb = self.position_embed(position).unsqueeze(1)
        instrument_emb = self.instrument_embed(instrument).unsqueeze(1)

        batch_size, window_size, _ = history.shape
        history_emb = self.history_embed(history.view(batch_size * window_size, self.n_features))
        history_emb = history_emb.view(batch_size, window_size, -1)

        # Concatenate along sequence dimension
        seq = torch.cat([market_emb, position_emb, instrument_emb, history_emb], dim=1)

        # Transformer processing
        transformer_out = self.transformer(seq)
        policy_out = self.fc_out(transformer_out.mean(dim=1))
        action_probs = self.softmax(policy_out)
        return action_probs

In [25]:
def train_worker(envs, model_config, chunk_size, epochs, lr, checkpoint_path):
    """Train the policy on the given environments."""
    # Set device to GPU if available, else CPU
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.xpu.is_available():
        device = torch.device("xpu")
    else:
        device = torch.device("cpu")

    # Initialize model and optimizer
    policy = TransformerPolicy(**model_config).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    # Load checkpoint if it exists
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        policy.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        best_avg_reward = checkpoint['best_avg_reward']
        print(f"Loaded checkpoint from epoch {checkpoint['epoch']} with best average reward {best_avg_reward:.2f}")
    else:
        best_avg_reward = -float("inf")

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        epoch_rewards = []

        for i in range(0, len(envs), chunk_size):
            chunk_envs = envs[i:i + chunk_size]
            total_reward = 0

            iterator = tqdm(chunk_envs, desc=f"Chunk {i//chunk_size + 1}")

            for env in iterator:
                obs, _ = env.reset()
                done = False
                episode_reward = 0
                log_probs = []
                rewards = []

                while not done:
                    action_probs = policy(obs).squeeze(0)
                    dist = Categorical(action_probs)
                    action = dist.sample()
                    log_prob = dist.log_prob(action)
                    next_obs, reward, done, _, _ = env.step(action.item())
                    log_probs.append(log_prob)
                    rewards.append(reward)
                    obs = next_obs
                    episode_reward += reward

                # Compute returns
                returns = []
                R = 0
                for r in reversed(rewards):
                    R = r + 0.99 * R
                    returns.insert(0, R)
                returns = torch.tensor(returns, dtype=torch.float32).to(device)
                returns = (returns - returns.mean()) / (returns.std() + 1e-8)

                # Compute policy loss
                policy_loss = 0
                for log_prob, R in zip(log_probs, returns):
                    policy_loss -= log_prob * R
                policy_loss /= len(rewards)

                # Update model parameters
                optimizer.zero_grad()
                policy_loss.backward()
                optimizer.step()

                total_reward += episode_reward

            avg_reward = total_reward / len(chunk_envs)
            epoch_rewards.append(avg_reward)
            print(f"Chunk {i//chunk_size + 1} Avg Reward: {avg_reward:.2f}")
            # Print instrument-specific metrics
            print(f"\nInstrument Metrics for Chunk {i//chunk_size + 1}")
            for env in chunk_envs:
                metrics = env.get_metrics()
                print(f"\nInstrument: {env.instrument_name}")
                for key, value in metrics.items():
                    print(f"{key}: {value}")

        avg_reward = sum(epoch_rewards) / len(epoch_rewards) if epoch_rewards else 0
        print(f"Epoch {epoch + 1} Average Reward: {avg_reward:.2f}")

        # Save checkpoint if improved
        if avg_reward > best_avg_reward:
            best_avg_reward = avg_reward
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': policy.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_avg_reward': best_avg_reward,
            }, checkpoint_path)
            print(f"New best model saved with average reward: {best_avg_reward:.2f}")

In [26]:
def train_grpo(envs, model_config, chunk_size, epochs=10, lr=1e-4, checkpoint_path="best_checkpoint.pth"):
    """Main training function for GPU or CPU."""
    train_worker(envs, model_config, chunk_size, epochs, lr, checkpoint_path)

    # Load the best model after training
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy = TransformerPolicy(**model_config).to(device)
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        policy.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model with average reward: {checkpoint['best_avg_reward']:.2f}")
    return policy

In [ ]:
# Set up environments (assuming config and TradingEnv are defined)
envs = []
for inst in config["instruments"]:
    env_config = config.copy()
    env_config.update(inst)
    env_config.pop("instruments")
    envs.append(TradingEnv(env_config))

# Infer n_features from a sample observation
sample_env = envs[0]
obs, _ = sample_env.reset()
model_config["n_features"] = obs["market"].shape[-1]

# Define checkpoint path
checkpoint_path = "/v3.1/best_checkpoint.pth"

# Set chunk_size based on the device
if torch.cuda.is_available():
    chunk_size = 10   # Moderate chunk size for CUDA
elif torch.xpu.is_available():
    chunk_size = 10   # Adjust based on performance on Intel GPU
else:
    chunk_size = 5    # Lower chunk size for CPU

# Train the policy
trained_policy = train_grpo(envs, model_config, chunk_size=chunk_size, epochs=5, checkpoint_path=checkpoint_path)

Epoch 1/5


Chunk 1: 100%|██████████| 5/5 [01:34<00:00, 18.80s/it]


Chunk 1 Avg Reward: 96.19

Instrument Metrics for Chunk 1

Instrument: BANK_NIFTY_240M
instrument: BANK_NIFTY_240M
initial_capital: ₹5.00L
final_capital: ₹6.24L
total_trades: 105
win_rate: 40.95%
total_reward: 80.65
max_drawdown: 0.3169074058532715
profit_factor: 1.09
smart_exit_rate: 0.0%
trailing_exit_rate: 33.33%

Instrument: BANKEX_240M
instrument: BANKEX_240M
initial_capital: ₹5.00L
final_capital: ₹6.47L
total_trades: 98
win_rate: 40.82%
total_reward: 95.99
max_drawdown: 0.18150807917118073
profit_factor: 1.23
smart_exit_rate: 0.0%
trailing_exit_rate: 32.65%

Instrument: FINNIFTY_240M
instrument: FINNIFTY_240M
initial_capital: ₹5.00L
final_capital: ₹5.33L
total_trades: 112
win_rate: 38.39%
total_reward: 98.86
max_drawdown: 0.3183678984642029
profit_factor: 1.05
smart_exit_rate: 0.0%
trailing_exit_rate: 38.39%

Instrument: NIFTY_240M
instrument: NIFTY_240M
initial_capital: ₹5.00L
final_capital: ₹8.20L
total_trades: 114
win_rate: 42.11%
total_reward: 110.03
max_drawdown: 0.245766654

Chunk 2: 100%|██████████| 5/5 [02:00<00:00, 24.11s/it]


Chunk 2 Avg Reward: 127.85

Instrument Metrics for Chunk 2

Instrument: BANK_NIFTY_180M
instrument: BANK_NIFTY_180M
initial_capital: ₹5.00L
final_capital: ₹7.88L
total_trades: 190
win_rate: 39.47%
total_reward: 127.65
max_drawdown: 0.36646586656570435
profit_factor: 1.16
smart_exit_rate: 0.0%
trailing_exit_rate: 29.47%

Instrument: BANKEX_180M
instrument: BANKEX_180M
initial_capital: ₹5.00L
final_capital: ₹5.30L
total_trades: 194
win_rate: 37.11%
total_reward: 142.61
max_drawdown: 0.2834491729736328
profit_factor: 1.03
smart_exit_rate: 0.0%
trailing_exit_rate: 26.29%

Instrument: FINNIFTY_180M
instrument: FINNIFTY_180M
initial_capital: ₹5.00L
final_capital: ₹4.47L
total_trades: 191
win_rate: 37.7%
total_reward: 124.75
max_drawdown: 0.3051615357398987
profit_factor: 0.96
smart_exit_rate: 0.0%
trailing_exit_rate: 31.94%

Instrument: NIFTY_180M
instrument: NIFTY_180M
initial_capital: ₹5.00L
final_capital: ₹5.41L
total_trades: 210
win_rate: 38.1%
total_reward: 119.55
max_drawdown: 0.463955

...

Chunk 8: 100%|██████████| 5/5 [19:05<00:00, 229.16s/it]


Chunk 8 Avg Reward: 546.74

Instrument Metrics for Chunk 8

Instrument: BANK_NIFTY_15M
instrument: BANK_NIFTY_15M
initial_capital: ₹5.00L
final_capital: ₹-7.63L
total_trades: 841
win_rate: 40.19%
total_reward: 600.91
max_drawdown: 2.041757583618164
profit_factor: 1.01
smart_exit_rate: 0.0%
trailing_exit_rate: 29.37%

Instrument: BANKEX_15M
instrument: BANKEX_15M
initial_capital: ₹5.00L
final_capital: ₹-4.26L
total_trades: 908
win_rate: 38.33%
total_reward: 623.06
max_drawdown: 1.7337357997894287
profit_factor: 0.91
smart_exit_rate: 0.0%
trailing_exit_rate: 31.06%

Instrument: FINNIFTY_15M
instrument: FINNIFTY_15M
initial_capital: ₹5.00L
final_capital: ₹-3.10L
total_trades: 849
win_rate: 40.4%
total_reward: 556.21
max_drawdown: 1.390397310256958
profit_factor: 0.99
smart_exit_rate: 0.0%
trailing_exit_rate: 28.98%

Instrument: NIFTY_15M
instrument: NIFTY_15M
initial_capital: ₹5.00L
final_capital: ₹3.03L
total_trades: 534
win_rate: 34.08%
total_reward: 272.05
max_drawdown: 0.4388095438480

Chunk 9: 100%|██████████| 5/5 [14:23<00:00, 172.74s/it]


Chunk 9 Avg Reward: 435.36

Instrument Metrics for Chunk 9

Instrument: BANK_NIFTY_10M
instrument: BANK_NIFTY_10M
initial_capital: ₹5.00L
final_capital: ₹3.09L
total_trades: 713
win_rate: 37.03%
total_reward: 501.53
max_drawdown: 0.5062493681907654
profit_factor: 0.93
smart_exit_rate: 0.0%
trailing_exit_rate: 28.75%

Instrument: BANKEX_10M
instrument: BANKEX_10M
initial_capital: ₹5.00L
final_capital: ₹-2.38L
total_trades: 918
win_rate: 38.67%
total_reward: 610.18
max_drawdown: 1.3235929012298584
profit_factor: 1.08
smart_exit_rate: 0.0%
trailing_exit_rate: 27.67%

Instrument: FINNIFTY_10M
instrument: FINNIFTY_10M
initial_capital: ₹5.00L
final_capital: ₹-2.15L
total_trades: 865
win_rate: 38.27%
total_reward: 578.24
max_drawdown: 1.2618014812469482
profit_factor: 1.12
smart_exit_rate: 0.0%
trailing_exit_rate: 27.63%

Instrument: NIFTY_10M
instrument: NIFTY_10M
initial_capital: ₹5.00L
final_capital: ₹3.06L
total_trades: 213
win_rate: 35.21%
total_reward: 124.76
max_drawdown: 0.45006704330

Chunk 10:  20%|██        | 1/5 [05:14<20:57, 314.37s/it]

In [ ]:
print("\nEnvironment Metrics:")
for env in envs:
    metrics = env.get_metrics()
    print("\nInstrument Metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value}")